##  Convert Excel data into MathProg data

In [1]:
from otoole import convert
from otoole import convert
convert('config.yaml', 'excel', 'csv', 'osemosys_otoole.xlsx', 'CSV')
convert('config.yaml', 'excel', 'datafile', 'osemosys_otoole.xlsx', 'osemosys.txt')

True

In [ ]:
! glpsol -m OSeMOSYS.txt -d osemosys_clicsand.txt --wlp simplicity.lp --nopresol

---
## Resolución con HiGHS

Usamos la API Python de `highspy` para resolver el LP de forma eficiente.

In [ ]:
import highspy

def solve_with_highs(
    lp_file: Path,
    sol_file: Path,
    time_limit: float = 7200.0,
    threads: int = 0,           # 0 = HiGHS elige automáticamente
    verbose: bool = True,
) -> dict:
    """
    Resuelve un LP con HiGHS vía API Python.
    Retorna un diccionario con el estado y la función objetivo.
    """
    h = highspy.Highs()

    # ── Opciones del solver ────────────────────────────────────────────────
    h.setOptionValue("solver",         "simplex")   # simplex o ipm
    h.setOptionValue("time_limit",     time_limit)
    h.setOptionValue("output_flag",    verbose)
    h.setOptionValue("log_to_console", verbose)
    if threads > 0:
        h.setOptionValue("threads", threads)

    # ── Leer modelo ───────────────────────────────────────────────────────
    print(f"Cargando LP: {lp_file}")
    t0 = time.time()
    status = h.readModel(str(lp_file))
    if status != highspy.HighsStatus.kOk:
        raise RuntimeError(f"Error al leer el LP: {status}")
    print(f"LP cargado en {time.time()-t0:.1f} s")

    # ── Info del modelo ───────────────────────────────────────────────────
    info = h.getModelDimension()
    n_cols = h.getNumCol()
    n_rows = h.getNumRow()
    print(f"  Variables   : {n_cols:,}")
    print(f"  Restricciones: {n_rows:,}")

    # ── Resolver ──────────────────────────────────────────────────────────
    print(f"\nResolviendo... ({time.strftime('%H:%M:%S')})")
    t1 = time.time()
    h.run()
    elapsed = time.time() - t1

    # ── Resultado ─────────────────────────────────────────────────────────
    model_status    = h.getModelStatus()
    sol_info        = h.getInfoValue("primal_solution_status")
    obj_value       = h.getInfoValue("objective_function_value")[1]
    simplex_iters   = h.getInfoValue("simplex_iteration_count")[1]

    print(f"\n─── Resultado HiGHS ─────────────────────────────────")
    print(f"  Estado del modelo : {model_status}")
    print(f"  Solución primal   : {sol_info}")
    print(f"  Función objetivo  : {obj_value:,.4f}")
    print(f"  Iteraciones Simplex: {simplex_iters:,}")
    print(f"  Tiempo de resolución: {elapsed:.1f} s")

    # ── Guardar solución ──────────────────────────────────────────────────
    h.writeSolution(str(sol_file), 1)  # style=1 → formato detallado
    print(f"\n  Solución guardada en: {sol_file}")

    return {
        "highs": h,
        "model_status": str(model_status),
        "obj_value": obj_value,
        "n_cols": n_cols,
        "n_rows": n_rows,
        "elapsed_s": elapsed,
    }

LP_FILE = Path("model.lp")
SOL_FILE = Path("model.sol")
result = solve_with_highs(
    lp_file   = LP_FILE,
    sol_file  = SOL_FILE,
    time_limit= 7200.0,
    verbose   = True,
)